# Google cell workload and power extraction

Run this notebook from top to bottom in Google Colab, or locally from the repository root.
It produces the measured workload curves and power fits used by the thesis.
The billing project defaults to `aeee-thesis`; change it below if needed.
BigQuery queries incur charges in that project.

Cells **a-d** supply the active experiment. Set `INCLUDE_UNUSED_CELLS = True` to also
extract **e-h** with the same workload filters. Those cells are unused in the current study.

Outputs are staged under `build/clusterdata2019/data/`, then packaged as
`build/clusterdata2019/clusterdata2019.zip`. This leaves the repository's committed
`data/` unchanged until you copy in the new files.

| Output inside the archive | Purpose |
|---|---|
| `data/cells/cell_x_tiers.csv` | Five-minute total, service, and batch CPU curves used by the factory |
| `data/cells/cell_x.csv` | Matching aggregate CPU curve |
| `data/power_model_params.json` | Per-cell CPU-to-power fits, plus the existing a-d pooled diagnostics |
| `data/power_model_scatter.csv` | Hourly a-d fitting observations for local inspection |

This replaces the three earlier extraction notebooks. The old job-duration,
machine-list, and synthetic-workload exports are no longer needed; their notebooks
remain in Git history. No EIA download or PPO run is part of this notebook.


## 1. Setup

Enable the BigQuery API in the billing project. Colab will prompt for Google
sign-in. For a local Jupyter kernel, first run `gcloud auth application-default login`
in a terminal using an account with BigQuery access to the project.


In [ ]:
%pip install -q google-cloud-bigquery db-dtypes pandas numpy


In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd
from google.cloud import bigquery

try:
    from google.colab import auth
except ImportError:
    IN_COLAB = False  # Use local Application Default Credentials.
else:
    IN_COLAB = True
    auth.authenticate_user()

PROJECT_ID = 'aeee-thesis'
INCLUDE_UNUSED_CELLS = False
ACTIVE_CELLS = ['a', 'b', 'c', 'd']
EXTRA_CELLS = ['e', 'f', 'g', 'h'] if INCLUDE_UNUSED_CELLS else []
CELLS = ACTIVE_CELLS + EXTRA_CELLS

OUTPUT_ROOT = Path('build/clusterdata2019')
DATA_DIR = OUTPUT_ROOT / 'data'
(DATA_DIR / 'cells').mkdir(parents=True, exist_ok=True)
generated_paths = []
client = bigquery.Client(project=PROJECT_ID)
print(f'Project: {PROJECT_ID}; cells: {", ".join(CELLS)}')
print(f'Output directory: {DATA_DIR.resolve()}')


## 2. Fixed capacity and five-minute workload curves

Reference capacity is the sum of each machine's largest reported CPU capacity.
Retain top-level usage rows with `end_time - start_time >= 300 seconds`, bucket
by `start_time`, and divide summed CPU usage by that fixed capacity.

Batch is usage with priority at most 115, using the highest priority among each
collection's `type = 0` events. Service is the remainder, including rows without
a matching priority. Preserve the existing clipping to `[0, 1]` and the five CSV
columns; the factory converts these curves to hourly arrivals.


In [ ]:
def get_cell_capacity(cell):
    """Get total CPU capacity for a cell."""
    query = f"""
    SELECT SUM(cpu_cap) AS cpu_capacity
    FROM (
        SELECT machine_id, MAX(capacity.cpus) AS cpu_cap
        FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.machine_events
        GROUP BY 1
    )
    """
    return float(client.query(query).to_dataframe()['cpu_capacity'].iloc[0])

# Pre-fetch capacities (reused by multiple datasets)
cell_capacities = {}
for cell in CELLS:
    cell_capacities[cell] = get_cell_capacity(cell)
    print(f"Cell {cell} CPU capacity: {cell_capacities[cell]:.2f}")


In [ ]:
for cell in CELLS:
    print(f'--- Cell {cell}: per-tier usage curves ---')
    ds = f'`google.com:google-cluster-data`.clusterdata_2019_{cell}'
    # priority lives on collection_events, not instance_usage -> join on collection_id.
    # Deferrable = no-SLO tiers = priority <= 115 (free <= 99, beb 100-115 per the
    # trace documentation v3, which corrects the 110-115 erratum in Tirmazi et al. 2020).
    query = f"""
    WITH cap AS (
        SELECT SUM(cpu_cap) AS cpu_capacity FROM (
            SELECT machine_id, MAX(capacity.cpus) AS cpu_cap
            FROM {ds}.machine_events GROUP BY 1
        )
    ),
    prio AS (
        SELECT collection_id, MAX(priority) AS priority
        FROM {ds}.collection_events
        WHERE type = 0
        GROUP BY collection_id
    )
    SELECT
        CAST(FLOOR(u.start_time / (1e6 * 300)) AS INT64) AS time_bucket,
        SUM(u.average_usage.cpus) / (SELECT cpu_capacity FROM cap) AS total_norm,
        SUM(IF(p.priority <= 115,
               u.average_usage.cpus, 0)) / (SELECT cpu_capacity FROM cap) AS batch_norm
    FROM {ds}.instance_usage u
    LEFT JOIN prio p USING (collection_id)
    WHERE (u.alloc_collection_id IS NULL OR u.alloc_collection_id = 0)
        AND (u.end_time - u.start_time) >= (5 * 60 * 1e6)
    GROUP BY 1
    ORDER BY 1
    """
    df = client.query(query).to_dataframe()
    df['timestep'] = df['time_bucket'] - df['time_bucket'].min()
    df['cpu_demand_norm'] = df['total_norm'].clip(0.0, 1.0)
    df['batch_demand_norm'] = df['batch_norm'].clip(0.0, 1.0).clip(upper=df['cpu_demand_norm'])
    df['service_demand_norm'] = df['cpu_demand_norm'] - df['batch_demand_norm']
    df['batch_share'] = (df['batch_demand_norm'] / df['cpu_demand_norm'].clip(lower=1e-9)).clip(0, 1)
    out = df[['timestep', 'cpu_demand_norm', 'batch_share',
              'service_demand_norm', 'batch_demand_norm']]
    path = DATA_DIR / 'cells' / f'cell_{cell}_tiers.csv'
    aggregate_path = DATA_DIR / 'cells' / f'cell_{cell}.csv'
    out.to_csv(path, index=False)
    out[['timestep', 'cpu_demand_norm']].to_csv(aggregate_path, index=False)
    generated_paths.extend([path, aggregate_path])
    bf = out['batch_demand_norm'].sum() / out['cpu_demand_norm'].sum()
    pm = out['batch_demand_norm'].max() / out['batch_demand_norm'].mean()
    print(f'  {len(out)} rows -> {path}   batch_fraction={bf:.3f}  batch peak/mean={pm:.2f}')

print('\nDone — measured per-tier curves (service + batch = measured aggregate).')


## 3. Hourly power observations

CPU utilization predicts measured power-domain utilization. The measurements
include data-center-floor cooling; see
[Google Data Center Power Trace, p. 2](https://raw.githubusercontent.com/google/cluster-data/master/power_trace_documentation.pdf#page=2).

Keep the existing power quality filter, average power by cell and hour, and
join on the trace's absolute hour index. CPU for the fit is the raw hourly
usage sum divided by `12 * reference capacity`, before workload clipping.

For a-d, retain memory in the fitting data and the existing finite-value filter
to reproduce the same fitted rows. Memory is used only in the pooled diagnostic;
the active per-cell model uses CPU alone.


In [ ]:
print('Extracting hourly power utilization...')
power_query = """
SELECT
    cell,
    CAST(FLOOR(time / (1e6 * 60 * 60)) AS INT64) AS hour_index,
    AVG(measured_power_util) AS avg_power_util
FROM `google.com:google-cluster-data`.`powerdata_2019.cell*`
WHERE NOT bad_measurement_data
    AND cell IN ('a', 'b', 'c', 'd')
GROUP BY 1, 2
ORDER BY 1, 2
"""
power_df = client.query(power_query).to_dataframe()
print(f'  {len(power_df)} power rows')


def get_cell_mem_capacity(cell):
    """Total (trace-normalized) memory capacity for a cell."""
    q = f"""
    SELECT SUM(mem_cap) AS mem_capacity FROM (
        SELECT machine_id, MAX(capacity.memory) AS mem_cap
        FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.machine_events
        GROUP BY 1
    )
    """
    return float(client.query(q).to_dataframe()['mem_capacity'].iloc[0])


# Hourly CPU *and* memory utilization per cell (memory is a candidate 2nd predictor)
all_cpu = []
for cell in ACTIVE_CELLS:
    print(f'  Hourly CPU+memory for cell {cell}...')
    cap = cell_capacities[cell]
    memcap = get_cell_mem_capacity(cell)
    q = f"""
    SELECT
        CAST(FLOOR(start_time / (1e6 * 60 * 60)) AS INT64) AS hour_index,
        SUM(average_usage.cpus) / (12 * {cap}) AS avg_cpu_util,
        SUM(average_usage.memory) / (12 * {memcap}) AS avg_mem_util
    FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.instance_usage
    WHERE (alloc_collection_id IS NULL OR alloc_collection_id = 0)
        AND (end_time - start_time) >= (5 * 60 * 1e6)
    GROUP BY 1
    ORDER BY 1
    """
    cpu_df = client.query(q).to_dataframe()
    cpu_df['cell'] = cell
    all_cpu.append(cpu_df)

cpu_combined = pd.concat(all_cpu, ignore_index=True)
merged = pd.merge(cpu_combined, power_df, on=['cell', 'hour_index'], how='inner')
valid = (
    np.isfinite(merged['avg_cpu_util'])
    & np.isfinite(merged['avg_mem_util'])
    & np.isfinite(merged['avg_power_util'])
)
merged = merged[valid].copy()


In [ ]:
cpu = merged['avg_cpu_util'].values
mem = merged['avg_mem_util'].values
pw = merged['avg_power_util'].values


def _fit(X, y):
    """Least-squares fit with intercept. X: (n, k) predictors. Returns (coefs, R2),
    coefs[0] = intercept."""
    A = np.column_stack([np.ones(len(y)), X])
    coef = np.linalg.lstsq(A, y, rcond=None)[0]
    pred = A @ coef
    ss_res = np.sum((y - pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return coef, float(1.0 - ss_res / ss_tot)


# 1) Pooled CPU-only diagnostic over a-d.
(intercept, slope), r2_cpu = _fit(cpu.reshape(-1, 1), pw)
intercept, slope = float(intercept), float(slope)

# 2) Pooled CPU + memory (multiple regression) — does memory absorb residual scatter?
coef_cm, r2_cm = _fit(np.column_stack([cpu, mem]), pw)

# 3) Per-cell CPU-only — each cell has a different machine mix → its own idle/slope.
per_cell = {}
for cell in ACTIVE_CELLS:
    mk = merged['cell'].values == cell
    if mk.sum() > 10:
        (a0, a1), r2c = _fit(cpu[mk].reshape(-1, 1), pw[mk])
        per_cell[cell] = {
            'idle_power': float(a0), 'slope': float(a1),
            'peak_power': float(a0 + a1), 'r_squared': r2c, 'n': int(mk.sum()),
        }

# 4) Binned diagnostic — is the *mean* relationship linear? (separates model quality
#    from point-level noise / the narrow CPU-utilization range of aggregate load).
bins = np.linspace(cpu.min(), cpu.max(), 21)
bi = np.digitize(cpu, bins)
bx, by = [], []
for b in range(1, 21):
    mm = bi == b
    if mm.sum() > 5:
        bx.append(cpu[mm].mean())
        by.append(pw[mm].mean())
bx, by = np.array(bx), np.array(by)
_, r2_binned = _fit(bx.reshape(-1, 1), by)

params = {
    # Pooled diagnostics retain their existing schema.
    'idle_power': intercept,
    'peak_power': intercept + slope,
    'slope': slope,
    'r_squared': float(r2_cpu),
    'description': ('Measured power-domain utilization (including floor cooling): '
                    'P = idle_power + slope * cpu_utilization'),
    # Additional diagnostics; the factory uses per_cell_cpu_model below.
    'cpu_util_range': [float(cpu.min()), float(cpu.max())],
    'binned_r_squared': float(r2_binned),
    'cpu_mem_model': {
        'idle_power': float(coef_cm[0]),
        'cpu_coef': float(coef_cm[1]),
        'mem_coef': float(coef_cm[2]),
        'r_squared': float(r2_cm),
        'description': 'P = idle_power + cpu_coef*cpu_util + mem_coef*mem_util',
    },
    'per_cell_cpu_model': per_cell,
}

print(f'\nPooled CPU-only:  idle={intercept:.4f} slope={slope:.4f}  R2={r2_cpu:.4f}')
print(f'Pooled CPU+mem:   R2={r2_cm:.4f}  (cpu={coef_cm[1]:.4f}, mem={coef_cm[2]:.4f})')
print(f'Binned-means R2:  {r2_binned:.4f}')
for cell, pc in per_cell.items():
    print(f'  cell {cell}: idle={pc["idle_power"]:.3f} slope={pc["slope"]:.3f} '
          f'R2={pc["r_squared"]:.4f} (n={pc["n"]})')


### Optional cells e-h and export

When enabled, e-h use their original CPU-only fit and finite CPU/power filter.
Their coefficients go into the same `per_cell_cpu_model` dictionary as a-d;
no separate coefficient file or merge step is needed. The pooled diagnostics
and scatter export remain a-d only.


In [ ]:
if EXTRA_CELLS:
    extra_power = client.query("""
    SELECT cell, CAST(FLOOR(time / (1e6*60*60)) AS INT64) AS hour_index,
           AVG(measured_power_util) AS avg_power_util
    FROM `google.com:google-cluster-data`.`powerdata_2019.cell*`
    WHERE NOT bad_measurement_data AND cell IN ('e','f','g','h')
    GROUP BY 1, 2
    """).to_dataframe()
    for cell in EXTRA_CELLS:
        cap = cell_capacities[cell]
        hourly_cpu = client.query(f"""
        SELECT CAST(FLOOR(start_time / (1e6*60*60)) AS INT64) AS hour_index,
               SUM(average_usage.cpus) / (12 * {cap}) AS avg_cpu_util
        FROM `google.com:google-cluster-data`.clusterdata_2019_{cell}.instance_usage
        WHERE (alloc_collection_id IS NULL OR alloc_collection_id = 0)
            AND (end_time - start_time) >= (5*60*1e6)
        GROUP BY 1
        """).to_dataframe()
        observations = hourly_cpu.merge(
            extra_power[extra_power['cell'] == cell], on='hour_index'
        )
        x = observations['avg_cpu_util'].to_numpy()
        y = observations['avg_power_util'].to_numpy()
        valid = np.isfinite(x) & np.isfinite(y)
        if valid.sum() < 2:
            raise ValueError(f'Cell {cell}: insufficient power-fit observations')
        (a0, a1), r2 = _fit(x[valid].reshape(-1, 1), y[valid])
        per_cell[cell] = {
            'idle_power': float(a0), 'slope': float(a1),
            'peak_power': float(a0 + a1), 'r_squared': r2,
            'n': int(valid.sum()),
        }
        print(f'Cell {cell}: idle={a0:.4f}, slope={a1:.4f}, R2={r2:.4f}')

if set(per_cell) != set(CELLS):
    raise ValueError('Missing per-cell power fits; check the hourly observations')

params_path = DATA_DIR / 'power_model_params.json'
params_path.write_text(json.dumps(params, indent=2, allow_nan=False) + '\n', encoding='utf-8')
scatter_path = DATA_DIR / 'power_model_scatter.csv'
pd.DataFrame({
    'cell': merged['cell'].values,
    'cpu_util': cpu,
    'mem_util': mem,
    'power_util': pw,
}).to_csv(scatter_path, index=False)
generated_paths.extend([params_path, scatter_path])


## 4. Check and download

The checks below verify the curve columns, spacing, service/batch totals, and
power coefficients before packaging. They do not rebuild the factory or run PPO.

Download the zip using Colab's Files panel if the automatic download is unavailable.
Review the extracted `data/` files, then copy them into the repository to use the
new extraction. Rebuild the factory afterward: changed inputs require a fresh
factory and new runs. Existing results remain tied to the committed data.


In [ ]:
expected_columns = [
    'timestep', 'cpu_demand_norm', 'batch_share',
    'service_demand_norm', 'batch_demand_norm',
]
for cell in CELLS:
    curve = pd.read_csv(DATA_DIR / 'cells' / f'cell_{cell}_tiers.csv')
    assert list(curve.columns) == expected_columns, f'Cell {cell}: wrong CSV columns'
    assert len(curve) > 0 and curve['timestep'].iloc[0] == 0, f'Cell {cell}: empty or shifted curve'
    assert np.all(np.diff(curve['timestep']) == 1), f'Cell {cell}: missing or duplicate buckets'
    assert len(curve) % 12 in (0, 1), f'Cell {cell}: incomplete hourly groups'
    assert np.isfinite(curve.to_numpy()).all(), f'Cell {cell}: nonfinite curve values'
    assert ((curve.iloc[:, 1:] >= 0) & (curve.iloc[:, 1:] <= 1)).all().all()
    np.testing.assert_allclose(
        curve['service_demand_norm'] + curve['batch_demand_norm'],
        curve['cpu_demand_norm'], rtol=0, atol=1e-12,
    )
    aggregate = pd.read_csv(DATA_DIR / 'cells' / f'cell_{cell}.csv')
    pd.testing.assert_frame_equal(aggregate, curve[['timestep', 'cpu_demand_norm']])
    model = params['per_cell_cpu_model'][cell]
    assert np.isfinite(list(model.values())).all(), f'Cell {cell}: nonfinite power fit'
    assert model['idle_power'] >= 0 and model['slope'] >= 0, f'Cell {cell}: invalid active power model'
    np.testing.assert_allclose(model['peak_power'], model['idle_power'] + model['slope'])
    print(f'Cell {cell}: {len(curve):,} five-minute rows; {model["n"]} fitted hours')

archive_path = OUTPUT_ROOT / 'clusterdata2019.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    # Package only this run's outputs, even if optional files from an earlier run remain.
    for path in sorted(set(generated_paths)):
        archive.write(path, arcname=path.relative_to(OUTPUT_ROOT).as_posix())
print(f'Validated {len(set(generated_paths))} files: {archive_path.resolve()}')

if IN_COLAB:
    from google.colab import files
    try:
        files.download(str(archive_path))
    except Exception as exc:
        print(f'Automatic download unavailable ({exc}); use the Colab Files panel.')
